In [1]:
import numpy as np
import numba


@numba.njit
def SD_flow(r0, drift, steps=100, eps_=1e-3, max_abs_r=10., bound=1e-3, dual = False):
    rlist = []
    r = r0
    if dual: eps = -eps_
    else: eps = eps_
    for _ in range(steps):
        dS = drift(r)
        dS_abs = np.abs(dS)
        if eps*dS_abs > bound: r += +bound/dS_abs * dS.conjugate()
        else: r += +eps * dS.conjugate()
        if np.abs(r) >= max_abs_r:
            break
        rlist.append(r)
    return np.array(rlist)



In [2]:

@numba.njit
def drift(z, sigma, lamb, pullback, mass_modification):
    sigma, lamb, pullback, mass_modification
    action = sigma/2*z**2+lamb/4*z**4
    drift = sigma*z+lamb*z**3
    # to prevent overflow
    if np.real(action-mass_modification/2*z**2)<0:
        return drift - pullback*np.exp(action-mass_modification*z**2/2)*(drift - mass_modification*z) / (1+pullback*np.exp(action-mass_modification/2*z**2))
    else: 
        return drift - pullback*(drift - mass_modification*z) / (np.exp(-action+mass_modification/2*z**2)+pullback)
    


In [3]:

from scipy.optimize import newton

def find_critical_points(action, x_range=[-5, 5], y_range=[-5, 5], x_n=100, y_n=100):
    x_vals = np.linspace(*x_range, x_n) 
    y_vals = np.linspace(*y_range, y_n)
    X, Y = np.meshgrid(x_vals, y_vals)
    Z = X+1j*Y

    points = []
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            try:
                res = newton(action, Z[i,j])
                points.append(res)
            except: pass #print(f"{Z[i,j]} did not work")
    points = np.array(np.unique(np.round(points, 8)))
    return points


In [4]:

def get_num_relevant(drift, x_range=[-4, 4], y_range=[-4, 4],
                       grid_points=100, steps=200000, nudge_amplitude=0.01, num_nudges=16,):
    
    critical = find_critical_points(drift, x_range=x_range, y_range=y_range, x_n=grid_points, y_n=grid_points)
    # print(f"found {len(critical)} points")

    relevant = []

    for cp in critical:
        num_nudges = 5
        angles = np.linspace(0+0.01, 2*np.pi+0.01, num_nudges, endpoint=False)
        perturbations = nudge_amplitude * np.exp(1j * angles)
        combined_sol = []  
            
        for pert in perturbations:  
            sol = SD_flow(cp + pert, drift, steps=steps, dual=True) 
            sol = sol[~np.isnan(sol)] 
            combined_sol.append(sol)  

        combined_sol = np.concatenate(combined_sol)

        if np.any(combined_sol.imag > 0) and np.any(combined_sol.imag < 0):
            relevant.append(cp)  

    return len(relevant)



In [5]:

import pandas as pd

# import os, json
# base_path = os.path.join(os.getenv("HOME"), "gitrepos", "clonscal_bak", "thimbles", "single_thimbles")
# os.makedirs(base_path, exist_ok=True)  # Create directory if it doesn't exist

# csv_path = os.path.join(base_path, f"single_thimbles.csv")
params = []

alpha_steps = 20
lambda_abs = 2
data = pd.DataFrame(columns=['mass_modification', 'pullback_upper' ,'pullback_lower', 'sigma_abs', 'sigma_phase' ,'lambda_abs'])

print(rf"This program uses a bisection algorithm to determine the critical r value (strength of Gaussian modification) for different values of a. The modification used: $r e^{{-\alpha z^2 / 2}}$")
for sigma in [-1+1j]:#, -1+2j, -1+3j, -1+4j]:
    for mass_modification in [1]:
        sigma_abs = np.abs(sigma)
        sigma_phase = np.angle(sigma)
            
        lower = 0
        upper = 500
        while np.abs(upper-lower) > 0.1:
            pullback = (upper+lower)/2

            drift_lambda = numba.njit(lambda z: drift(z, sigma, lambda_abs, pullback, mass_modification))
            num = get_num_relevant(drift_lambda)
            if num ==  1: 
                upper = pullback
            else: 
                lower = pullback
            # print(lower, upper)

        results = {}
        results['mass_modification'] = mass_modification
        results['pullback_upper'] = upper
        results['pullback_lower'] = lower
        results['sigma_abs'] = sigma_abs
        results['sigma_phase'] = sigma_phase
        results['lambda_abs'] = lambda_abs

        print(results)
        new_row_df = pd.DataFrame([results])
        data = pd.concat([data, new_row_df], ignore_index=True)
        print(data)

        # data.to_csv(csv_path, index=False)


This program uses a bisection algorithm to determine the critical r value (strength of Gaussian modification) for different values of a. The modification used: $r e^{-\alpha z^2 / 2}$
{'mass_modification': 1, 'pullback_upper': 7.080078125, 'pullback_lower': 7.01904296875, 'sigma_abs': 1.4142135623730951, 'sigma_phase': 2.356194490192345, 'lambda_abs': 2}
  mass_modification  pullback_upper  pullback_lower  sigma_abs  sigma_phase  \
0                 1        7.080078        7.019043   1.414214     2.356194   

  lambda_abs  
0          2  


/var/folders/d1/h9d5k7zx0c7555qbjvw2ck240000gn/T/ipykernel_63433/1056691143.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data, new_row_df], ignore_index=True)
